# Summarization Evaluator

Evaluates **summarization quality** using ROUGE metrics computed without external dependencies.

For each `(document, reference_summary)` pair the evaluator:
- Prompts the model to summarize the document
- Scores the generated summary against the reference using ROUGE-1, ROUGE-2, and ROUGE-L
- Reports compression ratio (summary words / document words)

**ROUGE scores:**
- **ROUGE-1**: unigram overlap — captures vocabulary coverage
- **ROUGE-2**: bigram overlap — captures phrase fluency
- **ROUGE-L**: longest common subsequence — captures sentence-level structure

All three scores are 0–1; higher is better. For abstractive summarization, ROUGE-L F1 ≥ 0.3 is generally considered good.

## 1. Imports

In [1]:
from bellmira.evaluators import ModelSummarizationEvaluator

c:\Users\luism\AppData\Local\pypoetry\FreshCache\virtualenvs\bellmira-4LtuUDYr-py3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Prepare (document, reference_summary) pairs

Each pair is a tuple of `(full_document_text, human_written_reference_summary)`.
The reference is the ground truth the model is scored against.

In [2]:
PAIRS = [
    (
        """O Millennium BCP é o maior banco privado em Portugal. Foi fundado em 1985 e tem a sua sede em Lisboa.
O banco oferece uma vasta gama de produtos e serviços financeiros, incluindo contas à ordem, crédito habitação,
cartões, seguros, investimentos e soluções digitais como o MBway. Em 2023, o banco registou lucros recordes
e continuou a expandir a sua presença no mercado. O banco tem operações em vários países, incluindo Polónia,
Moçambique e Angola.""",
        "O Millennium BCP é o maior banco privado português, fundado em 1985, com sede em Lisboa e presença internacional."
    ),
    (
        """Retrieval-Augmented Generation (RAG) is a technique that enhances large language models by incorporating
external knowledge at inference time. Instead of relying solely on parameters learned during training,
a RAG system retrieves relevant documents from a knowledge base and provides them as additional context
to the model. This approach significantly reduces hallucinations, improves factual accuracy, and allows
the model to answer questions about information that was not part of its training data.""",
        "RAG improves LLM accuracy by retrieving external documents at inference time, reducing hallucinations."
    ),
    (
        """In the beginning, God created the heavens and the earth. The earth was formless and empty, and darkness
covered the deep waters. And the Spirit of God was hovering over the surface of the waters.
Then God said, 'Let there be light,' and there was light. And God saw that the light was good.
Then he separated the light from the darkness. God called the light 'day' and the darkness 'night.'
And evening passed and morning came, marking the first day.""",
        "God created light and separated it from darkness on the first day of creation."
    ),
    (
        """Transformer models use self-attention mechanisms to process sequences in parallel, unlike recurrent
neural networks which process tokens sequentially. The attention mechanism computes relationships between
all pairs of tokens in a sequence simultaneously. This parallelism enables much faster training on modern
GPUs. The original Transformer architecture was introduced by Vaswani et al. in 2017 and has since become
the dominant paradigm for natural language processing tasks.""",
        "Transformers use parallel self-attention instead of sequential processing, enabling faster training than RNNs."
    ),
]

## 3. Configuration

In [3]:
MODEL_URL = "http://localhost:8080/"

## 4. Initialise the evaluator

In [4]:
evaluator = ModelSummarizationEvaluator(
    url=MODEL_URL,
    pairs=PAIRS,
    temperature=0.0,
    max_tokens=128,
)

print(f"Model: {evaluator.model_name}")

Model: Qwen/Qwen3.5-4B


In [5]:
evaluator._summarize("""O Millennium BCP é o maior banco privado em Portugal. Foi fundado em 1985 e tem a sua sede em Lisboa.
O banco oferece uma vasta gama de produtos e serviços financeiros, incluindo contas à ordem, crédito habitação,
cartões, seguros, investimentos e soluções digitais como o MBway. Em 2023, o banco registou lucros recordes
e continuou a expandir a sua presença no mercado. O banco tem operações em vários países, incluindo Polónia,
Moçambique e Angola.""")

('Fundado em 1985 e sediado em Lisboa, o Millennium BCP é o maior banco privado de Portugal, oferecendo uma ampla gama de produtos financeiros e soluções digitais como o MBway. O banco registou lucros recordes em 2023 enquanto expande as suas operações para vários países, incluindo Polónia, Moçambique e Angola.',
 1.9656593799591064,
 166,
 77)

In [6]:
from bellmira.llm_model.llm_model_client import ModelClient

model = ModelClient(base_url=MODEL_URL)
req = model.build_chat_request(
    user_prompt="""O Millennium BCP é o maior banco privado em Portugal. Foi fundado em 1985 e tem a sua sede em Lisboa.
O banco oferece uma vasta gama de produtos e serviços financeiros, incluindo contas à ordem, crédito habitação,
cartões, seguros, investimentos e soluções digitais como o MBway. Em 2023, o banco registou lucros recordes
e continuou a expandir a sua presença no mercado. O banco tem operações em vários países, incluindo Polónia,
Moçambique e Angola.""",
    system_prompt="You are a concise summarization assistant. "
        "When given a document, output only the summary — no preamble, no commentary.",
    model_name=model.get_model_name(),
    temperature=0,
    max_tokens=512,
    enable_thinking=False
)
response = model.send_request(req)
response.text

'{"id":"chatcmpl-8bee83951d5a2e7f","object":"chat.completion","created":1782496023,"model":"Qwen/Qwen3.5-4B","choices":[{"index":0,"message":{"role":"assistant","content":"O Millennium BCP, fundado em 1985 e sediado em Lisboa, é o maior banco privado de Portugal, oferecendo uma ampla gama de produtos financeiros e soluções digitais como o MBway. Com operações internacionais em Polónia, Moçambique e Angola, o banco registou lucros recordes em 2023 enquanto expande a sua presença no mercado.","refusal":null,"annotations":null,"audio":null,"function_call":null,"tool_calls":[],"reasoning":null},"logprobs":null,"finish_reason":"stop","stop_reason":null,"token_ids":null,"routed_experts":null}],"service_tier":null,"system_fingerprint":"vllm-0.23.0-4ea8bb9e","usage":{"prompt_tokens":154,"total_tokens":234,"completion_tokens":80,"prompt_tokens_details":null},"prompt_logprobs":null,"prompt_token_ids":null,"prompt_text":null,"kv_transfer_params":null}'

In [7]:
response.json()

{'id': 'chatcmpl-8bee83951d5a2e7f',
 'object': 'chat.completion',
 'created': 1782496023,
 'model': 'Qwen/Qwen3.5-4B',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'O Millennium BCP, fundado em 1985 e sediado em Lisboa, é o maior banco privado de Portugal, oferecendo uma ampla gama de produtos financeiros e soluções digitais como o MBway. Com operações internacionais em Polónia, Moçambique e Angola, o banco registou lucros recordes em 2023 enquanto expande a sua presença no mercado.',
    'refusal': None,
    'annotations': None,
    'audio': None,
    'function_call': None,
    'tool_calls': [],
    'reasoning': None},
   'logprobs': None,
   'finish_reason': 'stop',
   'stop_reason': None,
   'token_ids': None,
   'routed_experts': None}],
 'service_tier': None,
 'system_fingerprint': 'vllm-0.23.0-4ea8bb9e',
 'usage': {'prompt_tokens': 154,
  'total_tokens': 234,
  'completion_tokens': 80,
  'prompt_tokens_details': None},
 'prompt_logprobs': None,
 'p

## 5. Run evaluation

In [8]:
results = evaluator.evaluate()

for i, r in enumerate(results):
    if r.get("ROUGE1_F1") is None:
        print(f"\n[{i+1}] ERROR: {r.get('Error')}")
        continue
    print(f"\n[{i+1}] R1={r['ROUGE1_F1']:.3f}  R2={r['ROUGE2_F1']:.3f}  RL={r['ROUGEL_F1']:.3f}  "
          f"compression={r['Compression_ratio']:.3f}  latency={r['Latency']:.2f}s")
    print(f"     Reference:  {r['Reference_summary']}")
    print(f"     Generated:  {r['Generated_summary']}")


[1] R1=0.348  R2=0.239  RL=0.290  compression=0.667  latency=1.75s
     Reference:  O Millennium BCP é o maior banco privado português, fundado em 1985, com sede em Lisboa e presença internacional.
     Generated:  Fundado em 1985 e sediado em Lisboa, o Millennium BCP é o maior banco privado de Portugal, oferecendo uma ampla gama de produtos financeiros e soluções digitais como o MBway. O banco registou lucros recordes em 2023 enquanto expande as suas operações para vários países, incluindo Polónia, Moçambique e Angola.

[2] R1=0.259  R2=0.154  RL=0.222  compression=0.578  latency=1.23s
     Reference:  RAG improves LLM accuracy by retrieving external documents at inference time, reducing hallucinations.
     Generated:  Retrieval-Augmented Generation (RAG) enhances large language models by retrieving relevant external documents at inference time to serve as additional context. This method reduces hallucinations and improves factual accuracy, enabling the model to answer questions abo

## 6. Extract aggregate metrics

In [9]:
metrics = evaluator.extract_threshold_metrics(results)

print(f"Pairs evaluated:      {metrics['Pairs_evaluated']}")
print(f"Avg ROUGE-1 F1:       {metrics['Avg_ROUGE1_F1']}")
print(f"Avg ROUGE-2 F1:       {metrics['Avg_ROUGE2_F1']}")
print(f"Avg ROUGE-L F1:       {metrics['Avg_ROUGEL_F1']}")
print(f"Avg ROUGE-1 Precision:{metrics['Avg_ROUGE1_precision']}")
print(f"Avg ROUGE-1 Recall:   {metrics['Avg_ROUGE1_recall']}")
print(f"Avg compression:      {metrics['Avg_compression_ratio']}  (lower = more concise)")
print(f"Avg latency:          {metrics['Avg_latency']}s")
print(f"Avg prompt tokens:    {metrics['Avg_prompt_tokens']}")
print(f"Avg completion tok:   {metrics['Avg_completion_tokens']}")

Pairs evaluated:      4
Avg ROUGE-1 F1:       0.274
Avg ROUGE-2 F1:       0.1348
Avg ROUGE-L F1:       0.2138
Avg ROUGE-1 Precision:0.1787
Avg ROUGE-1 Recall:   0.6043
Avg compression:      0.6947  (lower = more concise)
Avg latency:          1.5577s
Avg prompt tokens:    151.0
Avg completion tok:   68.0


## 7. Customise the summarisation prompt

In [10]:
# Evaluate with a Portuguese-specific prompt
evaluator_pt = ModelSummarizationEvaluator(
    url=MODEL_URL,
    pairs=PAIRS[:2],
    system_prompt="És um assistente de sumarização conciso. Responde apenas com o resumo, sem introduções.",
    user_template="Resume o seguinte texto em duas frases:\n\n{document}",
    temperature=0.0,
    max_tokens=128,
)

res_pt = evaluator_pt.evaluate()
for r in res_pt:
    print(f"R1={r.get('ROUGE1_F1')}  RL={r.get('ROUGEL_F1')}")
    print(f"  {r.get('Generated_summary')}\n")

R1=0.4179  RL=0.4179
  O Millennium BCP é o maior banco privado de Portugal, fundado em 1985 com sede em Lisboa, que oferece uma ampla gama de produtos financeiros e soluções digitais. Em 2023, o banco registou lucros recordes e expandiu as suas operações para vários países, incluindo Polónia, Moçambique e Angola.

R1=0.0  RL=0.0
  A técnica Retrieval-Augmented Generation (RAG) melhora modelos de linguagem grandes ao incorporar conhecimento externo durante a inferência, recuperando documentos relevantes de uma base de conhecimento para fornecer contexto adicional. Essa abordagem reduz alucinações, aumenta a precisão factual e permite que o modelo responda a perguntas sobre informações não presentes nos dados de treinamento.



## 8. Log to MLflow

In [ ]:
import mlflow

mlflow.set_experiment("summarization_evaluation")
with mlflow.start_run(run_name="Qwen35-sum"):
    loggable = {k: v for k, v in metrics.items() if isinstance(v, (int, float)) and v is not None}
    mlflow.log_metrics(loggable)
    mlflow.log_param("model_url", MODEL_URL)
    mlflow.log_param("pairs_count", len(PAIRS))
    print("Logged:", loggable)

Logged: {'Pairs_evaluated': 4, 'Avg_ROUGE1_F1': 0.274, 'Avg_ROUGE2_F1': 0.1348, 'Avg_ROUGEL_F1': 0.2138, 'Avg_ROUGE1_precision': 0.1787, 'Avg_ROUGE1_recall': 0.6043, 'Avg_compression_ratio': 0.6947, 'Avg_latency': 1.5577, 'Avg_prompt_tokens': 151.0, 'Avg_completion_tokens': 68.0}
